# Analisis Geoespacial de Hotspots Comerciales: Wendy's San Salvador
Prototipo academico de Machine Learning (K-Means + KNN con distancia Haversine).

In [1]:
import pandas as pd
import numpy as np
import folium
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import LeaveOneOut

df = pd.read_csv('wendys_san_salvador_limpio.csv')
print(f'Total de sucursales cargadas: {len(df)}')
df.head()

Total de sucursales cargadas: 11


,osm_type,osm_id,nombre,marca,tipo_local,latitud,longitud,direccion,ciudad,horario,telefono,sitio_web
0,node,6052795498,Wendy's,Wendy's,fast_food,13.689027,-89.240304,NaN,NaN,NaN,NaN,NaN
1,node,9089891045,Wendy's,Wendy's,fast_food,13.697944,-89.151035,NaN,NaN,NaN,NaN,https://www.wendys.com.sv/
2,node,10212633849,Wendy's,Wendy's,fast_food,13.664425,-89.184422,NaN,NaN,NaN,NaN,NaN
3,way,378380416,Wendy's,Wendy's,fast_food,13.701655,-89.231375,Paseo General Escalón,San Salvador,NaN,NaN,NaN
4,way,382289412,Wendy's,Wendy's,fast_food,13.683516,-89.230466,NaN,NaN,NaN,NaN,NaN


## 1. Agrupamiento Espacial con K-Means
Evaluacion de Inercia y Coeficiente de Silueta para identificar polos comerciales.

In [2]:
X = df[['latitud', 'longitud']].to_numpy()
for k in range(2, min(7, len(df))):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X)
    print(f'K={k} | Inercia={km.inertia_:.6f} | Silueta={silhouette_score(X, labels):.4f}')

kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
df['cluster'] = kmeans.fit_predict(X)
conteo = df['cluster'].value_counts()
promedio = conteo.mean()
df['categoria_densidad'] = np.where(df['cluster'].map(conteo) > promedio, 'Alta Concentracion', 'Baja Concentracion')
df[['nombre', 'latitud', 'longitud', 'cluster', 'categoria_densidad']]

K=2 | Inercia=0.004706 | Silueta=0.5614
K=3 | Inercia=0.002748 | Silueta=0.5114
K=4 | Inercia=0.001287 | Silueta=0.4906
K=5 | Inercia=0.000883 | Silueta=0.3381
K=6 | Inercia=0.000565 | Silueta=0.2944


,nombre,latitud,longitud,cluster,categoria_densidad
0,Wendy's,13.689027,-89.240304,0,Alta Concentracion
1,Wendy's,13.697944,-89.151035,1,Alta Concentracion
2,Wendy's,13.664425,-89.184422,2,Baja Concentracion
3,Wendy's,13.701655,-89.231375,0,Alta Concentracion
4,Wendy's,13.683516,-89.230466,0,Alta Concentracion
5,Wendy's,13.700693,-89.221425,0,Alta Concentracion
6,Wendy's,13.705652,-89.248891,0,Alta Concentracion
7,Wendy's,13.709331,-89.209095,0,Alta Concentracion
8,Wendy's,13.699537,-89.151525,1,Alta Concentracion
9,Wendy's,13.712618,-89.185751,1,Alta Concentracion


## 2. Clasificacion Supervisada con KNN Haversine y Validacion Leave-One-Out

In [3]:
X_rad = np.radians(df[['latitud', 'longitud']].to_numpy())
y = np.array(df['categoria_densidad'].tolist())
loo = LeaveOneOut()

for k in range(1, len(df), 2):
    aciertos = 0
    for train_idx, test_idx in loo.split(X_rad):
        knn = KNeighborsClassifier(n_neighbors=k, metric='haversine')
        knn.fit(X_rad[train_idx], y[train_idx])
        if knn.predict(X_rad[test_idx])[0] == y[test_idx][0]:
            aciertos += 1
    print(f'k={k} | Exactitud LOO: {aciertos/len(df):.2%}')

k=1 | Exactitud LOO: 90.91%
k=3 | Exactitud LOO: 90.91%
k=5 | Exactitud LOO: 90.91%
k=7 | Exactitud LOO: 90.91%
k=9 | Exactitud LOO: 90.91%


## 3. Evaluacion de Nueva Ubicacion Propuesta

In [4]:
knn_final = KNeighborsClassifier(n_neighbors=3, metric='haversine')
knn_final.fit(X_rad, y)

punto_nuevo = np.radians([[13.7015, -89.231]])
pred = knn_final.predict(punto_nuevo)[0]
dist_rad, indices = knn_final.kneighbors(punto_nuevo)
dist_km = dist_rad[0] * 6371.0

print('Prediccion:', pred)
for idx, d in zip(indices[0], dist_km):
    print(f"{df.loc[idx, 'nombre']} -> {d:.3f} km")

Prediccion: Alta Concentracion
Wendy's -> 0.044 km
Wendy's -> 1.038 km
Wendy's -> 1.713 km
